In [1]:
# %% [markdown]
# # 16 — Sparse KAN v4 (Full Re-Search, G=3 for Layer 0)
#
# ═══════════════════════════════════════════════════════════════════════════
# ARCHITECTURAL UPDATE: REDUCED CAPACITY IN LAYER 0
# ═══════════════════════════════════════════════════════════════════════════
#
# This notebook conducts a full, from-scratch 70-trial hyperparameter
# search (30 trials No-L1, 40 trials With-L1) for the fully corrected KAN.
#
# The architecture inherits the mathematical fixes from V2/V3:
#   - PyTorch advanced indexing initialization fix applied (Bug A fixed).
#   - BatchNorm configured with `affine=False` and `eps=1e-5` (Bug B fixed).
#   - Grid boundaries safely protected by torch.clamp.
#
# NEW IN V4: Layer 0's B-spline grid size has been reduced from G=14 to
# G=3. Because Layer 0 processes the massive 1699 feature input, giving
# it 14 knot intervals created too much hyper-expressivity early in
# the network. Shrinking G to 3 forces Layer 0 to act as a stiffer,
# more generalized feature aggregator, pushing the heavy non-linear work
# to the deeper layers (which remain at G=14).
#
# Uses `sparse_kan_v4.py`.
# Results are saved to a clean `sparse_kan_v4` directory.

# %%
# ── COLAB SETUP ──
!pip install -q git+https://github.com/Blealtan/efficient-kan.git optuna

from google.colab import drive
drive.mount("/content/drive")

import sys
sys.path.insert(0, "/content/drive/MyDrive/Thesis/Code")

# %%
import json
import numpy as np
import pandas as pd
import random
import time
import torch
import optuna
from pathlib import Path

from data_utils import load_split, get_dataloaders, get_device, load_theme_assignment
from training import train_model, save_checkpoint
from evaluation import (
    evaluate_model, save_predictions, compute_calibration,
    load_predictions, run_full_backtest,
)

# IMPORTANT: Importing from the V4 file
from sparse_kan_v4 import SparseKAN, sparse_kan_edge_l1

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


# ═══════════════════════════════════════════════════════════════════════════
# ACTIVATION MONITORING (Including Clamp Tracking)
# ═══════════════════════════════════════════════════════════════════════════

def make_activation_callback(probe_batch, device):
    @torch.no_grad()
    def callback(model, epoch):
        model.eval()
        x0 = probe_batch.to(device)

        x1 = model.layer0(x0)
        x1n_preclamp = model.bn1(x1)
        x1n = torch.clamp(x1n_preclamp, min=-5.0, max=5.0)

        x2 = model.layer1(x1n)
        x2n_preclamp = model.bn2(x2)
        x2n = torch.clamp(x2n_preclamp, min=-5.0, max=5.0)

        model.train()

        clamp1_rate = (x1n_preclamp.abs() > 5.0).float().mean().item()
        clamp2_rate = (x2n_preclamp.abs() > 5.0).float().mean().item()

        print(f"    [activations @ epoch {epoch}]  "
              f"L1 pre-BN max|x|={x1.abs().max().item():7.2f} std={x1.std().item():6.3f}  "
              f"post-BN(pre-clamp) max|x|={x1n_preclamp.abs().max().item():5.2f} "
              f"std={x1n_preclamp.std().item():5.3f}  clamp_rate={clamp1_rate:.3%}  |  "
              f"L2 pre-BN max|x|={x2.abs().max().item():7.2f} std={x2.std().item():6.3f}  "
              f"post-BN(pre-clamp) max|x|={x2n_preclamp.abs().max().item():5.2f} "
              f"std={x2n_preclamp.std().item():5.3f}  clamp_rate={clamp2_rate:.3%}")
    return callback


# ═══════════════════════════════════════════════════════════════════════════
# SANITY CHECK — Verify V4 Init Fix + Split Grids
# ═══════════════════════════════════════════════════════════════════════════

print("=" * 90)
print("SANITY CHECK: Confirming V4 Init Fix + eps=1e-5 + Split Grid Sizes")
print("=" * 90)

N_DUMMY = 200
_cols = [f"f{i}" for i in range(N_DUMMY)]
_fake_tax = pd.DataFrame({
    "column":        _cols,
    "subtheme_id":   ["01_01"]*5 + [f"02_{i:02d}" for i in range(N_DUMMY - 5)],
    "subtheme_name": ["SubA"]*5  + ["SubB"]*(N_DUMMY - 5),
    "theme_id":      ["01"]*5    + ["02"]*(N_DUMMY - 5),
    "theme_name":    ["ThemeX"]*5 + ["ThemeY"]*(N_DUMMY - 5),
})

# Note the split grid arguments here to test V4
_m = SparseKAN.from_taxonomy(_fake_tax, _cols, grid_size_0=3, grid_size_12=5, 
                             spline_order=3, grid_range=[-1, 1])

# Check eps & affine
assert abs(_m.bn1.eps - 1e-5) < 1e-9, f"bn1.eps={_m.bn1.eps}, expected 1e-5. STOP."
assert abs(_m.bn2.eps - 1e-5) < 1e-9, f"bn2.eps={_m.bn2.eps}, expected 1e-5. STOP."
assert _m.bn1.affine is False, f"bn1.affine={_m.bn1.affine}, expected False. STOP."
assert _m.bn2.affine is False, f"bn2.affine={_m.bn2.affine}, expected False. STOP."
print(f"  ✓ eps=1e-5, affine=False on both bn1/bn2")

# Check Init Size (The advanced indexing bug fix)
_active = _m.layer0.mask[0].bool()
_fan_in = int(_active.sum().item())
assert _fan_in == 5, f"fixture built wrong: fan_in={_fan_in}, expected 5"
_bound = 1.0 / _fan_in**0.5
for _pname, _W in [("base_weight",   _m.layer0.base_weight.data),
                   ("spline_scaler", _m.layer0.spline_scaler.data)]:
    _ratio = _W[0][_active].abs().max().item() / _bound
    assert _ratio > 0.5, f"{_pname} ratio={_ratio:.3f} -- STALE FILE. STOP."
    print(f"  ✓ init fix on {_pname}: ratio = {_ratio:.3f} (fan_in={_fan_in}, expect ~0.83)")

assert _m.verify_masking(), "masking broken at construction. STOP."
_m.train()
assert _m(torch.randn(16, N_DUMMY)).shape == (16, 1)
print("  ✓ forward pass OK in train mode (batch=16)")

del _fake_tax, _cols, _m, _active, _fan_in, _bound, N_DUMMY

print("\n" + "=" * 90)
print("SANITY CHECK PASSED -- Safe to proceed with full V4 sweep")
print("=" * 90)


# ═══════════════════════════════════════════════════════════════════════════
# CONFIGURATION
# ═══════════════════════════════════════════════════════════════════════════

SPLITS_DIR  = Path("/content/drive/MyDrive/Thesis/Data/Stage_5_Model_Ready/04_splits")
THEMES_DIR  = Path("/content/drive/MyDrive/Thesis/Data/Stage_5_Model_Ready/05_themes")
HUBER_DELTA_PATH = Path("/content/drive/MyDrive/Thesis/Data/Stage_5_Model_Ready/03_targets/huber_delta.json")

# ── WRITE to the NEW V4 directory ──
RESULTS_DIR = Path("/content/drive/MyDrive/Thesis/Data/Results/Dense_vs_Sparse_KAN_v2/sparse_kan_v4")

DATASETS     = ["agg_full_moments", "agg_means"]
TARGET_TYPES = ["binary", "continuous"]
ALL_SPLITS   = ["Split_A", "Split_B", "Split_C", "Split_D"]

SEEDS = [42, 123, 456]

# ── Split Architecture ──
GRID_SIZE_0  = 3     # Stiffer aggregation for L0
GRID_SIZE_12 = 14    # Full capacity for L1/L2
SPLINE_ORDER = 3
GRID_RANGE   = [-5.5, 5.5]

ACTIVATION_PROBE_N = 2048

N_TRIALS       = 40   # with L1
N_TRIALS_NO_L1 = 30   # without L1

with open(HUBER_DELTA_PATH) as f:
    HUBER_DELTAS = json.load(f)["deltas"]

def get_huber_delta(split_name):
    return HUBER_DELTAS[f"{split_name}/market"]


# ═══════════════════════════════════════════════════════════════════════════
# REPRODUCIBILITY
# ═══════════════════════════════════════════════════════════════════════════

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# ═══════════════════════════════════════════════════════════════════════════
# LOAD TAXONOMIES
# ═══════════════════════════════════════════════════════════════════════════

print("Loading taxonomies...")
taxonomy_dfs = {}
for ds in DATASETS:
    df = load_theme_assignment(ds, THEMES_DIR)
    taxonomy_dfs[ds] = df
    print(f"  {ds}: {len(df)} features, "
          f"{df['subtheme_id'].nunique()} subthemes, "
          f"{df['theme_id'].nunique()} themes")

device = get_device()


# ═══════════════════════════════════════════════════════════════════════════
# MODEL FACTORIES FOR OPTUNA
# ═══════════════════════════════════════════════════════════════════════════

def make_model_factory_no_l1(feature_cols, taxonomy_df, huber_delta, target_type):
    def factory(trial):
        lr           = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
        weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-2, log=True)
        batch_size   = trial.suggest_categorical("batch_size", [64, 128, 256])

        model = SparseKAN.from_taxonomy(
            taxonomy_df, feature_cols,
            grid_size_0=GRID_SIZE_0, grid_size_12=GRID_SIZE_12, 
            spline_order=SPLINE_ORDER, grid_range=GRID_RANGE,
        )

        train_kwargs = {
            "lr":            lr,
            "weight_decay": weight_decay,
            "n_epochs":      300,
            "patience":      20,
        }
        if target_type == "continuous":
            train_kwargs["huber_delta"] = huber_delta

        return model, train_kwargs
    return factory


def make_model_factory_with_l1(feature_cols, taxonomy_df, huber_delta, target_type):
    def factory(trial):
        lr           = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
        weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-2, log=True)
        batch_size   = trial.suggest_categorical("batch_size", [64, 128, 256])
        reg_weight   = trial.suggest_float("reg_weight", 1e-7, 1e-3, log=True)

        model = SparseKAN.from_taxonomy(
            taxonomy_df, feature_cols,
            grid_size_0=GRID_SIZE_0, grid_size_12=GRID_SIZE_12, 
            spline_order=SPLINE_ORDER, grid_range=GRID_RANGE,
        )

        train_kwargs = {
            "lr":            lr,
            "weight_decay": weight_decay,
            "reg_fn":        sparse_kan_edge_l1,
            "reg_weight":    reg_weight,
            "n_epochs":      300,
            "patience":      20,
        }
        if target_type == "continuous":
            train_kwargs["huber_delta"] = huber_delta

        return model, train_kwargs
    return factory


def robust_optimize(study, objective, n_trials, max_retries=3):
    """Retry study.optimize() on transient SQLite/Drive I/O errors."""
    for attempt in range(max_retries):
        try:
            study.optimize(objective, n_trials=n_trials, show_progress_bar=True)
            return
        except Exception as e:
            if "disk I/O error" in str(e) or "OperationalError" in str(e):
                print(f"  ⚠ Transient storage error (attempt {attempt+1}/{max_retries}): {e}")
                time.sleep(5)
                continue
            raise  
    raise RuntimeError(f"study.optimize failed after {max_retries} retries -- persistent Drive issue.")


# ═══════════════════════════════════════════════════════════════════════════
# SINGLE EXPERIMENT
# ═══════════════════════════════════════════════════════════════════════════

def run_single_experiment(split_name, dataset, target_type, device, seed, seed_results_dir):
    model_name = f"sparse_kan_{dataset}"

    print(f"\n{'─'*60}")
    print(f"  seed={seed} / {dataset} / {split_name} / {target_type}")
    print(f"{'─'*60}")

    data         = load_split(split_name, dataset, SPLITS_DIR)
    feature_cols = data["feature_cols"]
    taxonomy_df  = taxonomy_dfs[dataset]

    n_pos      = data["y_train"].sum()
    n_neg      = len(data["y_train"]) - n_pos
    pos_weight = torch.tensor([n_neg / max(n_pos, 1)], dtype=torch.float32)

    huber_delta = get_huber_delta(split_name) if target_type == "continuous" else None
    metric_name = "AUC" if target_type == "binary" else "R2"

    rng = np.random.default_rng(seed)
    n_avail = data["X_train"].shape[0]
    probe_idx = rng.choice(n_avail, size=min(ACTIVATION_PROBE_N, n_avail), replace=False)
    activation_probe = torch.tensor(data["X_train"][probe_idx], dtype=torch.float32)

    def run_optuna_phase(phase_name, factory, n_trials):
        study_name = f"{model_name}_{target_type}_{split_name}_{phase_name}_seed{seed}"
        study_path = seed_results_dir / "optuna" / f"{study_name}.db"
        study_path.parent.mkdir(parents=True, exist_ok=True)

        study = optuna.create_study(
            study_name=study_name,
            storage=f"sqlite:///{study_path}",
            direction="maximize",
            load_if_exists=True,
            sampler=optuna.samplers.TPESampler(seed=seed),
        )

        def objective(trial):
            model, train_kwargs = factory(trial)
            batch_size = trial.params["batch_size"]

            loaders = get_dataloaders(
                split_name, dataset, SPLITS_DIR,
                target_type=target_type, batch_size=batch_size,
            )

            result = train_model(
                model=model,
                train_loader=loaders["train"],
                val_loader=loaders["val"],
                device=device,
                target_type=target_type,
                pos_weight=pos_weight if target_type == "binary" else None,
                verbose=False,
                **train_kwargs,
            )
            return result["best_val_metric"]

        optuna.logging.set_verbosity(optuna.logging.WARNING)
        existing  = sum(1 for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE)
        remaining = max(0, n_trials - existing)

        if remaining > 0:
            print(f"  {phase_name}: {remaining} trials ({existing} already complete)")
            robust_optimize(study, objective, remaining)
        else:
            print(f"  {phase_name}: {existing} trials already complete — skipping search")

        print(f"  {phase_name} best {metric_name}: {study.best_value:.4f}  params: {study.best_params}")
        return study

    factory_no_l1 = make_model_factory_no_l1(feature_cols, taxonomy_df, huber_delta, target_type)
    study_no_l1 = run_optuna_phase("no_L1", factory_no_l1, N_TRIALS_NO_L1)

    factory_l1 = make_model_factory_with_l1(feature_cols, taxonomy_df, huber_delta, target_type)
    study_l1 = run_optuna_phase("with_L1", factory_l1, N_TRIALS)

    no_l1_wins = study_no_l1.best_value >= study_l1.best_value

    if no_l1_wins:
        best_params = study_no_l1.best_params
        best_val    = study_no_l1.best_value
        use_l1      = False
        print(f"\n  → No-L1 wins (seed={seed}, {metric_name}={best_val:.4f})")
    else:
        best_params = study_l1.best_params
        best_val    = study_l1.best_value
        use_l1      = True
        print(f"\n  → With-L1 wins (seed={seed}, {metric_name}={best_val:.4f})")

    # ── FINAL TRAINING ──
    batch_size = best_params.get("batch_size", 128)
    loaders    = get_dataloaders(
        split_name, dataset, SPLITS_DIR,
        target_type=target_type, batch_size=batch_size,
    )

    model = SparseKAN.from_taxonomy(
        taxonomy_df, feature_cols,
        grid_size_0=GRID_SIZE_0, grid_size_12=GRID_SIZE_12, 
        spline_order=SPLINE_ORDER, grid_range=GRID_RANGE,
    )
    activation_callback = make_activation_callback(activation_probe, device)

    final_train_kwargs = {
        "lr":              best_params["lr"],
        "weight_decay":    best_params["weight_decay"],
        "pos_weight":      pos_weight if target_type == "binary" else None,
        "n_epochs":        300,
        "patience":        20,
        "verbose":         True,
        "log_every":       20,
        "epoch_callback":  activation_callback,
    }
    if use_l1:
        final_train_kwargs["reg_fn"]     = sparse_kan_edge_l1
        final_train_kwargs["reg_weight"] = best_params["reg_weight"]
    if target_type == "continuous":
        final_train_kwargs["huber_delta"] = huber_delta

    result = train_model(
        model=model,
        train_loader=loaders["train"],
        val_loader=loaders["val"],
        device=device,
        target_type=target_type,
        **final_train_kwargs,
    )

    assert model.verify_masking(), "Masked weights non-zero after final training completed."

    all_metrics = {}
    for part in ["train", "val", "test"]:
        metrics = evaluate_model(
            model, loaders[part], device, target_type,
            y_true_binary=data[f"y_{part}"] if target_type == "continuous" else None,
        )

        if target_type == "binary":
            cal = compute_calibration(metrics["y_true"], metrics["y_prob"])
            metrics["ece"] = cal["ece"]

        save_predictions(
            model_name=model_name,
            split_name=split_name,
            target_type=target_type,
            part=part,
            dates=data[f"dates_{part}"],
            returns=data[f"returns_{part}"],
            metrics=metrics,
            hyperparameters=(
                {**best_params, "used_l1": use_l1, "seed": seed, "huber_delta": huber_delta}
                if part == "test" else None
            ),
            results_dir=seed_results_dir,
            y_true_binary=data[f"y_{part}"] if target_type == "continuous" else None,
        )
        all_metrics[part] = metrics

    ckpt_dir = seed_results_dir / "checkpoints"
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    save_checkpoint(
        model=model,
        train_result=result,
        hyperparameters={**best_params, "used_l1": use_l1, "seed": seed, "huber_delta": huber_delta},
        model_config={
            "type":                "SparseKAN_Masked_BN_v4",
            "dataset":             dataset,
            "target_type":         target_type,
            "n_features":          data["n_features"],
            "n_subthemes":         model.n_subthemes,
            "n_themes":            model.n_themes,
            "grid_size_0":         GRID_SIZE_0,
            "grid_size_12":        GRID_SIZE_12,
            "spline_order":        SPLINE_ORDER,
            "grid_range":          GRID_RANGE,
            "eps":                 1e-5,
            "active_edges":        model.count_active_edges(),
            "active_parameters":   model.count_active_parameters(),
        },
        path=ckpt_dir / f"{model_name}_{target_type}_{split_name}.pt",
    )

    if target_type == "binary":
        print(f"\n  Results (seed={seed}):")
        print(f"    Train AUC: {all_metrics['train']['auc']:.4f}")
        print(f"    Val AUC:   {all_metrics['val']['auc']:.4f}")
        print(f"    Test AUC:  {all_metrics['test']['auc']:.4f}")
        print(f"    Gap:       {all_metrics['train']['auc'] - all_metrics['test']['auc']:+.4f}")
    else:
        print(f"\n  Results (seed={seed}, huber_delta={huber_delta:.4f}):")
        print(f"    Train R2:  {all_metrics['train']['r2']:.4f}")
        print(f"    Val R2:    {all_metrics['val']['r2']:.4f}")
        print(f"    Test R2:   {all_metrics['test']['r2']:.4f}")
        print(f"    Derived AUC: {all_metrics['test']['derived_auc']:.4f}")

    return {
        "best_params": best_params,
        "used_l1":     use_l1,
        "metrics":     all_metrics,
        "best_epoch":  result["best_epoch"],
        "total_time":  result["total_time"],
    }


# %% [markdown]
# ## Run All Experiments (3 Seeds × 16 Configurations)

# %%
configs    = len(DATASETS) * len(TARGET_TYPES) * len(ALL_SPLITS)
total_runs = len(SEEDS) * configs

print("=" * 70)
print(f"  SPARSE KAN V4 (FULL RE-SEARCH): {total_runs} runs")
print(f"  G=3 for Layer 0, G=14 for Layers 1/2.")
print(f"  Writing NEW results to: {RESULTS_DIR}")
print("=" * 70)

all_results       = []
best_params_store = {}
completed         = 0
failed            = 0
total_start       = time.time()

for seed in SEEDS:
    set_seed(seed)
    seed_results_dir = RESULTS_DIR / f"seed_{seed}"

    print(f"\n\n{'═'*70}")
    print(f"  SEED {seed} — saving to {seed_results_dir}")
    print(f"{'═'*70}")

    for dataset in DATASETS:
        for target_type in TARGET_TYPES:
            for split_name in ALL_SPLITS:
                try:
                    exp = run_single_experiment(
                        split_name, dataset, target_type, device,
                        seed=seed,
                        seed_results_dir=seed_results_dir,
                    )

                    all_results.append({
                        "seed":    seed,
                        "dataset": dataset,
                        "split":   split_name,
                        "target":  target_type,
                        "used_l1": exp["used_l1"],
                        **{f"test_{k}": v for k, v in exp["metrics"]["test"].items()
                           if not isinstance(v, np.ndarray)},
                        "best_epoch": exp["best_epoch"],
                        "time_s":     exp["total_time"],
                    })

                    key = (seed, dataset, target_type, split_name)
                    best_params_store[key] = {**exp["best_params"], "used_l1": exp["used_l1"]}
                    completed += 1

                    elapsed = time.time() - total_start
                    rate = elapsed / completed
                    remaining_est = rate * (total_runs - completed)
                    print(f"\n  ✓ Completed {completed}/{total_runs}  "
                          f"({elapsed/60:.0f}min elapsed, ~{remaining_est/60:.0f}min remaining)")

                except Exception as e:
                    failed += 1
                    print(f"\n  ✗ FAILED ({failed}): seed={seed} "
                          f"{dataset}/{split_name}/{target_type}: {e}")
                    import traceback
                    traceback.print_exc()
                    continue

total_time = time.time() - total_start
print(f"\n\n{'='*70}")
print(f"  FINISHED: {completed}/{total_runs} completed, {failed} failed")
print(f"  Total time: {total_time/60:.1f} minutes")
print(f"{'='*70}")


# %% [markdown]
# ## Cross-Seed Summary

# %%
if all_results:
    results_df = pd.DataFrame(all_results)
    raw_path = RESULTS_DIR / "all_seeds_raw.csv"
    results_df.to_csv(raw_path, index=False)
    print(f"Raw results saved to {raw_path}")

    binary_df = results_df[results_df["target"] == "binary"]
    print("\nBinary Test AUC (mean ± std across seeds):")
    if "test_auc" in binary_df.columns and len(binary_df) > 0:
        agg = binary_df.groupby(["dataset", "split"])["test_auc"].agg(["mean", "std"])
        print(agg.to_string())

    cont_df = results_df[results_df["target"] == "continuous"]
    print("\nContinuous Test R² (mean ± std across seeds):")
    if "test_r2" in cont_df.columns and len(cont_df) > 0:
        agg = cont_df.groupby(["dataset", "split"])["test_r2"].agg(["mean", "std"])
        print(agg.to_string())


# %% [markdown]
# ## File Inventory

# %%
print("\n" + "=" * 70)
print("  SAVED FILES (V4 -- on Google Drive)")
print("=" * 70)

for seed in SEEDS:
    seed_dir = RESULTS_DIR / f"seed_{seed}"
    print(f"\n  ── seed_{seed}/ ──")
    for subdir in ["predictions", "metrics", "checkpoints", "optuna"]:
        d = seed_dir / subdir
        if d.exists():
            files = list(d.glob("sparse_kan_*"))
            print(f"    {subdir}/: {len(files)} files")
        else:
            print(f"    {subdir}/: (not yet created)")


# %% [markdown]
# ## Backtests (Seed-Averaged Signal)

# %%
def load_averaged_predictions(model_name, split_name, target_type, part, seeds, results_dir):
    signals = []
    returns = None
    for seed in seeds:
        seed_dir = results_dir / f"seed_{seed}"
        loaded = load_predictions(model_name=model_name, split_name=split_name,
                                  target_type=target_type, part=part, results_dir=seed_dir)
        preds = loaded["predictions"]
        if returns is None:
            returns = preds["daily_return"].values
        signals.append(preds["y_prob"].values if target_type == "binary" else preds["y_pred"].values)
    return returns, np.mean(np.stack(signals, axis=0), axis=0)

def _make_json_safe(obj):
    if isinstance(obj, dict): return {k: _make_json_safe(v) for k, v in obj.items()}
    if isinstance(obj, pd.DataFrame): return obj.reset_index().to_dict(orient="records")
    if isinstance(obj, (np.floating, np.integer)): return obj.item()
    if isinstance(obj, np.ndarray): return obj.tolist()
    return obj

print("\n" + "=" * 70)
print("  SPARSE KAN V4 — BACKTESTS (seed-averaged signal)")
print("=" * 70)

backtest_dir = RESULTS_DIR / "backtests"
backtest_dir.mkdir(parents=True, exist_ok=True)
backtest_rows    = []
backtest_records = {}

for dataset in DATASETS:
    for split_name in ALL_SPLITS:
        model_name = f"sparse_kan_{dataset}"

        val_ret,  val_sig  = load_averaged_predictions(model_name, split_name, "binary", "val",  SEEDS, RESULTS_DIR)
        test_ret, test_sig = load_averaged_predictions(model_name, split_name, "binary", "test", SEEDS, RESULTS_DIR)
        bt_binary = run_full_backtest(val_returns=val_ret, val_signal=val_sig, test_returns=test_ret, test_signal=test_sig,
                                      go_cash_when="above", model_name=f"{model_name}_v4 (binary)", split_name=split_name)

        val_ret,  val_sig  = load_averaged_predictions(model_name, split_name, "continuous", "val",  SEEDS, RESULTS_DIR)
        test_ret, test_sig = load_averaged_predictions(model_name, split_name, "continuous", "test", SEEDS, RESULTS_DIR)
        bt_continuous = run_full_backtest(val_returns=val_ret, val_signal=val_sig, test_returns=test_ret, test_signal=test_sig,
                                          go_cash_when="below", model_name=f"{model_name}_v4 (continuous)", split_name=split_name)

        key = f"{dataset}/{split_name}"
        backtest_records[key] = {"binary": bt_binary, "continuous": bt_continuous}

        for target_type, bt in [("binary", bt_binary), ("continuous", bt_continuous)]:
            for strategy_name, strategy_key in [("simple", "simple"), ("risk_scaled", "risk_scaled")]:
                bt_result = bt[strategy_key]
                backtest_rows.append({
                    "dataset": dataset, "split": split_name,
                    "target_type": target_type, "strategy": strategy_name,
                    "sharpe": bt_result["sharpe"], "sortino": bt_result["sortino"],
                    "annual_return": bt_result["annual_return"], "max_drawdown": bt_result["max_drawdown"],
                    "buy_hold_sharpe": bt_result["buy_hold_sharpe"],
                })

backtest_summary_df = pd.DataFrame(backtest_rows)
backtest_summary_path = backtest_dir / "backtest_summary.csv"
backtest_summary_df.to_csv(backtest_summary_path, index=False)
print(f"\n  Backtest summary saved to {backtest_summary_path}")

backtest_json_path = backtest_dir / "backtest_full_results.json"
with open(backtest_json_path, "w") as f:
    json.dump(_make_json_safe(backtest_records), f, indent=2, default=str)
print(f"  Full backtest results saved to {backtest_json_path}")

# %%
print("All experiments complete. Disconnecting runtime...")
from google.colab import runtime
runtime.unassign()

Mounted at /content/drive
PyTorch: 2.11.0+cu128
CUDA: True
GPU: Tesla T4
SANITY CHECK: Confirming V4 Init Fix + eps=1e-5 + Split Grid Sizes
  ✓ eps=1e-5, affine=False on both bn1/bn2
  ✓ init fix on base_weight: ratio = 0.607 (fan_in=5, expect ~0.83)
  ✓ init fix on spline_scaler: ratio = 0.698 (fan_in=5, expect ~0.83)
  ✓ forward pass OK in train mode (batch=16)

SANITY CHECK PASSED -- Safe to proceed with full V4 sweep
Loading taxonomies...
  agg_full_moments: 1699 features, 331 subthemes, 13 themes
  agg_means: 574 features, 128 subthemes, 13 themes
Device: Tesla T4 (CUDA)
  SPARSE KAN V4 (FULL RE-SEARCH): 48 runs
  G=3 for Layer 0, G=14 for Layers 1/2.
  Writing NEW results to: /content/drive/MyDrive/Thesis/Data/Results/Dense_vs_Sparse_KAN_v2/sparse_kan_v4


══════════════════════════════════════════════════════════════════════
  SEED 42 — saving to /content/drive/MyDrive/Thesis/Data/Results/Dense_vs_Sparse_KAN_v2/sparse_kan_v4/seed_42
══════════════════════════════════════════════

[I 2026-08-16 00:32:58,591] A new study created in RDB with name: sparse_kan_agg_full_moments_binary_Split_A_no_L1_seed42


  no_L1: 30 trials (0 already complete)


  0%|          | 0/30 [00:00<?, ?it/s]

  no_L1 best AUC: 0.8474  params: {'lr': 0.0016738085788752138, 'weight_decay': 3.6138942712165278e-06, 'batch_size': 256}
  with_L1: 40 trials (0 already complete)


...

  with_L1 best R2: 0.2849  params: {'lr': 0.005907501032777252, 'weight_decay': 0.0010876016839281872, 'batch_size': 256, 'reg_weight': 3.4760835831289967e-06}

  → With-L1 wins (seed=456, R2=0.2849)
  Epoch    1 | Train Huber 0.8204 | Val Huber 1.3892  MSE 4.0886  R² -0.4479 | LR 5.9e-03
    [activations @ epoch 1]  L1 pre-BN max|x|=   6.14 std= 0.417  post-BN(pre-clamp) max|x|= 7.99 std=0.584  clamp_rate=0.052%  |  L2 pre-BN max|x|=   1.99 std= 0.233  post-BN(pre-clamp) max|x|= 3.36 std=0.386  clamp_rate=0.000%
  Epoch   20 | Train Huber 0.1851 | Val Huber 1.2316  MSE 3.3164  R² -0.1745 | LR 1.5e-03
    [activations @ epoch 20]  L1 pre-BN max|x|=   5.31 std= 0.393  post-BN(pre-clamp) max|x|=18.11 std=0.991  clamp_rate=0.301%  |  L2 pre-BN max|x|=   2.74 std= 0.420  post-BN(pre-clamp) max|x|= 6.30 std=0.991  clamp_rate=0.173%
  Early stop at epoch 23. Best val R²: 0.1302 at epoch 3
  Training complete in 3.5s

  Results (seed=456, huber_delta=2.4230):
    Train R2:  0.4958
    Val R2:

  0%|          | 0/30 [00:00<?, ?it/s]

  no_L1 best R2: 0.2856  params: {'lr': 0.000866179717633783, 'weight_decay': 0.00019054457596092667, 'batch_size': 128}
  with_L1: 40 trials (0 already complete)


  0%|          | 0/40 [00:00<?, ?it/s]

  with_L1 best R2: 0.1912  params: {'lr': 0.000377182869269302, 'weight_decay': 0.0003226036853479407, 'batch_size': 128, 'reg_weight': 0.0005882471239285141}

  → No-L1 wins (seed=456, R2=0.2856)
  Epoch    1 | Train Huber 1.1872 | Val Huber 1.5182  MSE 3.1545  R² -1.8072 | LR 8.7e-04
    [activations @ epoch 1]  L1 pre-BN max|x|=   5.30 std= 0.404  post-BN(pre-clamp) max|x|=10.55 std=0.775  clamp_rate=0.150%  |  L2 pre-BN max|x|=   2.31 std= 0.265  post-BN(pre-clamp) max|x|= 5.99 std=0.616  clamp_rate=0.015%
  Epoch   20 | Train Huber 0.3363 | Val Huber 0.7059  MSE 1.4199  R² -0.2636 | LR 4.3e-04
    [activations @ epoch 20]  L1 pre-BN max|x|=   5.28 std= 0.395  post-BN(pre-clamp) max|x|=18.76 std=0.986  clamp_rate=0.322%  |  L2 pre-BN max|x|=   3.28 std= 0.283  post-BN(pre-clamp) max|x|= 7.10 std=0.986  clamp_rate=0.124%
  Early stop at epoch 29. Best val R²: 0.1430 at epoch 9
  Training complete in 7.7s

  Results (seed=456, huber_delta=2.4998):
    Train R2:  0.4461
    Val R2:   